In [0]:
# Step 1: Set up the tables we want to test
from pyspark.sql import functions as F
from datetime import datetime, timezone

catalog = spark.sql("SELECT current_catalog()").first()[0]
schema_name = "retail_lakehouse"

silver_table = f"{catalog}.{schema_name}.streaming_orders_silver"
daily_table = f"{catalog}.{schema_name}.streaming_gold_daily_sales"
product_table = f"{catalog}.{schema_name}.streaming_gold_product_performance"
regional_table = f"{catalog}.{schema_name}.streaming_gold_regional_sales"

test_results = []
test_run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

print("Test run:", test_run_id)

In [0]:
# Step 2: Record test results in one place
def run_test(test_name, test_function):
    try:
        test_function()
        status = "PASS"
        message = "Test completed successfully"
    except Exception as error:
        status = "FAIL"
        message = str(error)[:500]

    test_results.append(
        (test_run_id, test_name, status, message)
    )

    print(f"{status}: {test_name}")
    if status == "FAIL":
        print("Reason:", message)

In [0]:
# Step 3: Load Silver data and check its basic quality
def test_silver_table_exists():
    assert spark.catalog.tableExists(silver_table), (
        f"Missing table: {silver_table}"
    )

run_test("silver_table_exists", test_silver_table_exists)

In [0]:
# Step 4: Check that order IDs are present
def test_order_ids_not_null():
    df = spark.table(silver_table)
    assert df.filter(F.col("order_id").isNull()).count() == 0, (
        "Silver contains null order IDs"
    )

run_test("order_ids_not_null", test_order_ids_not_null)

In [0]:
# Step 5: Check that each order ID appears only once
def test_order_ids_unique():
    df = spark.table(silver_table)

    duplicate_count = (
        df.groupBy("order_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    assert duplicate_count == 0, (
        f"Found {duplicate_count} duplicated order IDs"
    )

run_test("order_ids_unique", test_order_ids_unique)

In [0]:
# Step 6: Check quantity, price and date values
def test_valid_order_values():
    df = spark.table(silver_table)

    invalid_count = df.filter(
        F.col("order_date").isNull()
        | F.col("product_id").isNull()
        | F.col("quantity").isNull()
        | (F.col("quantity") <= 0)
        | F.col("unit_price").isNull()
        | (F.col("unit_price") < 0)
    ).count()

    assert invalid_count == 0, (
        f"Found {invalid_count} rows with invalid values"
    )

run_test("valid_order_values", test_valid_order_values)

In [0]:
# Step 7: Check that the Gold tables exist
def test_gold_tables_exist():
    for table_name in [daily_table, product_table, regional_table]:
        assert spark.catalog.tableExists(table_name), (
            f"Missing Gold table: {table_name}"
        )

run_test("gold_tables_exist", test_gold_tables_exist)

In [0]:
# Step 8: Check daily revenue against the Silver source
def test_daily_revenue_matches_source():
    source_df = (
        spark.table(silver_table)
        .filter(F.lower(F.trim(F.col("status"))) == "completed")
    )

    expected_df = source_df.groupBy("order_date").agg(
        F.round(
            F.sum(F.col("quantity") * F.col("unit_price")), 2
        ).alias("expected_revenue")
    )

    actual_df = spark.table(daily_table).select(
        "order_date",
        F.col("total_revenue").alias("actual_revenue")
    )

    comparison_df = (
        expected_df.alias("e")
        .join(actual_df.alias("a"), "order_date", "full")
        .filter(
            F.col("expected_revenue").isNull()
            | F.col("actual_revenue").isNull()
            | (
                F.abs(
                    F.coalesce(F.col("expected_revenue"), F.lit(0.0))
                    - F.coalesce(F.col("actual_revenue"), F.lit(0.0))
                ) > 0.01
            )
        )
    )

    mismatches = comparison_df.count()
    assert mismatches == 0, (
        f"Daily revenue mismatch on {mismatches} dates"
    )

run_test("daily_revenue_matches_source", test_daily_revenue_matches_source)

In [0]:
# Step 9: Display the results from this test run
results_df = spark.createDataFrame(
    test_results,
    ["test_run_id", "test_name", "status", "message"]
)

display(results_df.orderBy("status", "test_name"))

failed_tests = sum(
    1 for _, _, status, _ in test_results if status == "FAIL"
)

print("Total tests:", len(test_results))
print("Failed tests:", failed_tests)
print("Overall status:", "PASS" if failed_tests == 0 else "FAIL")

In [0]:
# Step 10: Save test results for future review
test_history_table = f"{catalog}.{schema_name}.automated_test_history"

if not spark.catalog.tableExists(test_history_table):
    results_df.write.format("delta").saveAsTable(test_history_table)
else:
    results_df.write.format("delta").mode("append").saveAsTable(
        test_history_table
    )

print("Test history saved:", test_history_table)

In [0]:
# Step 11: Mark the notebook run as failed when a test fails
if failed_tests > 0:
    raise RuntimeError(
        f"Automated testing failed: {failed_tests} test(s) need attention."
    )

print("All automated tests passed.")